# C-01 Ollama 직접 호출

이 노트북은 프레임워크를 거의 쓰지 않고 `requests`로 Ollama HTTP API를 직접 호출하는 가장 단순한 방식입니다.

## 무엇을 확인하나

- 로컬 Ollama 서버가 실제로 떠 있는지 확인합니다.
- 설치된 모델 목록을 조회하고, 사용할 모델이 없으면 실행 전에 명확한 오류를 냅니다.
- 이메일 분석 프롬프트를 직접 만들고 `/api/chat`에 전달합니다.
- 모델 응답을 문자열로 받은 뒤 Pydantic 모델로 검증합니다.

## 이해 포인트

Ollama는 로컬에서 LLM을 실행해 주는 서버입니다. 이 방식은 LangChain, LlamaIndex 같은 중간 계층 없이 Ollama의 REST API를 그대로 사용합니다. 그래서 동작 원리를 이해하기 쉽고 의존성이 적지만, 프롬프트 조립, 오류 처리, JSON 검증, 재시도 같은 주변 기능을 직접 구현해야 합니다.

## 언제 적합한가

- 작은 실험이나 API 동작 확인
- 외부 프레임워크 의존성을 줄이고 싶을 때
- LLM 호출 흐름을 정확히 통제하고 싶을 때

## 한계

- 프롬프트 체인, 도구 호출, RAG, 워크플로우 같은 기능은 직접 만들어야 합니다.
- 모델 응답이 항상 스키마를 지킨다는 보장이 없어서 후처리 검증이 필요합니다.

참고: `predicted_email_intent`, `predicted_email_importance` 같은 enum 값은 시스템 내부 분류 라벨이라 영어로 유지했습니다. 사용자에게 보여 줄 때는 별도 매핑으로 한글 표시명을 붙이면 됩니다.

In [ ]:
import json
import os
from typing import Literal

import requests
from pydantic import BaseModel, Field, ValidationError

# Ollama는 기본적으로 11434 포트에서 HTTP API를 제공합니다.
# 환경변수로 바꾸면 다른 서버나 컨테이너의 Ollama에도 연결할 수 있습니다.
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
DEFAULT_MODEL = "llama3.2:latest"
MODEL = os.getenv("OLLAMA_MODEL", DEFAULT_MODEL)

# /api/tags는 로컬에 설치된 모델 목록을 반환합니다.
# 모델명이 틀리면 /api/chat에서 404가 날 수 있으므로 먼저 확인합니다.
models_response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
models_response.raise_for_status()
available_models = [model["name"] for model in models_response.json().get("models", [])]

if MODEL not in available_models:
    raise ValueError(
        f"Ollama 모델 {MODEL!r}이 설치되어 있지 않습니다. "
        f"OLLAMA_MODEL을 다음 중 하나로 설정하세요: {', '.join(available_models)}"
    )

MODEL

In [ ]:
class EmailAnalysisResult(BaseModel):
    # LLM이 반환해야 하는 JSON 구조입니다.
    # Literal은 허용 가능한 분류 라벨을 제한해 잘못된 응답을 빨리 찾게 해 줍니다.
    # predicted_email_intent 라벨은 임시 업무 분류 체계입니다. 실제 운영 전 반드시 사용자 검토가 필요합니다.
    # - inquiry: 견적/납기/제품 문의처럼 아직 발주가 확정되지 않은 요청
    # - order: 구매 발주서, PO, 주문 확정처럼 실제 주문 처리로 이어지는 요청
    # - service: 클레임, 고장, 누수, 긴급 지원처럼 서비스/AS 대응이 필요한 요청
    # - technical: 도면, 사양, 기술 검토, 호환성 확인처럼 엔지니어링 판단이 필요한 요청
    # - other: 위 기준으로 분류하기 어렵거나 추가 업무 유형 정의가 필요한 메일
    # TODO: 실제 고객 메일 샘플을 보고 라벨 이름, 개수, 정의를 확정해야 합니다.
    predicted_email_intent: Literal["inquiry", "order", "service", "technical", "other"]
    # predicted_email_importance 라벨도 임시 우선순위 체계입니다. SLA, 업무 프로세스, 사용자 화면 정책에 맞게 조정해야 합니다.
    # - low: 참고/일반 정보성으로 즉시 처리가 필요하지 않은 메일
    # - normal: 통상 처리 기한 안에 대응하면 되는 일반 업무 메일
    # - high: 납기, 견적 마감, 고객 영향 등으로 우선 확인이 필요한 메일
    # - urgent: 긴급 수리, 선박 운항 영향, 즉시 회신 요구처럼 지연 시 손실이 큰 메일
    # TODO: predicted_email_importance는 단순 감정/단어가 아니라 실제 처리 SLA와 연결해 정의해야 합니다.
    predicted_email_importance: Literal["low", "normal", "high", "urgent"]
    # extracted_key_information은 모델이 이메일에서 추출한 핵심 업무 정보입니다.
    # 예: 견적번호, PO 번호, 제품명, 수량, 납기일, 선박명 등.
    # TODO: 실제 필드 목록이 확정되면 dict가 아니라 별도 Pydantic 모델로 바꾸는 것이 좋습니다.
    extracted_key_information: dict = Field(default_factory=dict)
    # predicted_assignee_area는 실제 개인 담당자라기보다 임시 담당 영역/팀 후보입니다.
    # 예: sales_team, service_team, technical_team, order_management 등.
    # TODO: 실제 사용자/팀/라우팅 규칙이 정리되면 assignee_user_id, assignee_team_id와 분리할지 결정해야 합니다.
    predicted_assignee_area: str
    # prediction_reasoning은 위 예측값을 낸 근거 설명입니다.
    # TODO: 실제 UI에 노출할지, 내부 감사/디버깅 용도로만 저장할지 결정해야 합니다.
    prediction_reasoning: str
    # predicted_needs_human_review는 AI가 사람 검토 필요성을 예측한 값입니다.
    # 최종 검토 상태가 아니며, 서비스 계층에서 정책/신뢰도/오류 여부와 함께 확정해야 합니다.
    predicted_needs_human_review: bool


# 실제 메일 DB 대신 분석 흐름을 확인하기 위한 한글 샘플입니다.
sample_email = {
    "subject": "펌프 예비품 견적 요청",
    "sender": "customer@example.com",
    "body": "선박 정기 수리에 사용할 펌프 예비품 견적을 요청드립니다. 가격과 납기일을 긴급히 회신 부탁드립니다.",
    "attachments": ["펌프_예비품_RFQ.pdf"],
}

In [ ]:
# 직접 호출 방식에서는 프롬프트 문자열도 직접 책임집니다.
# ensure_ascii=False를 사용해 한글 이메일 본문을 사람이 읽기 쉬운 형태로 넣습니다.
prompt = f"""
선박 부품 제조사 업무 이메일을 분석하세요.
아래 스키마 필드에 맞는 유효한 JSON만 반환하세요:
- predicted_email_intent: inquiry, order, service, technical, other
- predicted_email_importance: low, normal, high, urgent
- extracted_key_information: object
- predicted_assignee_area: string
- prediction_reasoning: string
- predicted_needs_human_review: boolean

이메일:
{json.dumps(sample_email, ensure_ascii=False, indent=2)}
"""

# Ollama native chat API 요청 형식입니다.
# format="json"은 모델이 JSON 형태로 답하도록 유도하지만, 최종 검증은 다음 셀에서 다시 합니다.
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": prompt}],
    "stream": False,
    "format": "json",
    "options": {"temperature": 0},
}

response = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload, timeout=120)
try:
    response.raise_for_status()
except requests.HTTPError as exc:
    raise RuntimeError(f"Ollama 요청 실패: {response.status_code} {response.text}") from exc

content = response.json()["message"]["content"]
print(content)

In [ ]:
# LLM 응답은 문자열이므로, 실제 애플리케이션에서 쓰기 전에 스키마 검증을 거칩니다.
# 검증이 실패하면 프롬프트 개선, 재시도, 사람 검토 같은 정책을 붙일 수 있습니다.
try:
    result = EmailAnalysisResult.model_validate_json(content)
    print(result.model_dump_json(indent=2))
except ValidationError as exc:
    print(exc)